# Module 04 — Day 1: Introduction to DeepEval for LLM Testing

**Course:** AI Testing APR Cohort  
**Module:** 04 — DeepEval & LLM-as-a-Judge  
**Day:** 1 of 5  

---

## Table of Contents

1. [What Problem Does DeepEval Solve?](#1-what-problem-does-deepeval-solve)
2. [Install Check](#2-install-check)
3. [Setup: Environment & LLM Client](#3-setup-environment--llm-client)
4. [The `LLMTestCase` Dataclass](#4-the-llmtestcase-dataclass)
5. [Create Your First Test Case](#5-create-your-first-test-case)
6. [AnswerRelevancyMetric — The First Judge](#6-answerrelevancymetric--the-first-judge)
7. [Measure a Test Case](#7-measure-a-test-case)
8. [How LLM-as-a-Judge Works (Under the Hood)](#8-how-llm-as-a-judge-works-under-the-hood)
9. [`assert_test()` — The pytest Integration Point](#9-assert_test--the-pytest-integration-point)
10. [A Real pytest Test with a Live LLM Call](#10-a-real-pytest-test-with-a-live-llm-call)
11. [Reading Metric Reasons](#11-reading-metric-reasons)
12. [Try It: Parametrize Over 3 Prompts](#12-try-it-parametrize-over-3-prompts)
13. [Day 1 Summary](#13-day-1-summary)

## 1. What Problem Does DeepEval Solve?

### The Limits of `assert` for AI Output

When you test regular software, you can do this:

```python
assert add(2, 3) == 5   # deterministic — always right or always wrong
```

But LLMs produce **natural language** — and two correct answers rarely look identical:

```python
response = ask_llm("What is the capital of France?")
assert response == "Paris"   # FAILS if the LLM says "The capital is Paris."
```

Worse, `assert` has **zero ability** to detect:
- Hallucinations (the model made something up)
- Irrelevant answers (technically words, but not on topic)
- Toxic or biased language
- Missing context from a retrieval step (RAG)

### The Analogy: A Peer Reviewer for Your AI

Think of it like academic peer review. When a researcher submits a paper, you don't run `assert paper == correct_paper`. Instead, **an expert reviewer reads the paper** and scores it on criteria: Is it relevant? Is it accurate? Is it well-reasoned?

**DeepEval does the same thing for your LLM's outputs.** It sends your model's response to a second LLM (the "judge") along with a rubric. The judge reads both the question and the answer, then returns:
- A **score** (0.0 to 1.0)
- A **reason** explaining the score
- A **pass/fail** decision based on a threshold you set

```
Your LLM's answer  ──►  Judge LLM  ──►  Score + Reason + Pass/Fail
     + rubric
```

DeepEval packages all of this into pytest-compatible helpers so your AI tests look and run exactly like your regular unit tests.

## 2. Install Check

Before anything else, confirm DeepEval is installed in your environment. If this cell throws a `ModuleNotFoundError`, run `pip install deepeval` in your terminal and restart the kernel.

In [ ]:
# Quick sanity check — import deepeval and print its version
import deepeval

print(deepeval.__version__)

## 3. Setup: Environment & LLM Client

### Why we use `.env` files

API keys are secrets — we never hard-code them in notebooks. Instead we store them in a `.env` file (which is git-ignored) and load them at runtime with `python-dotenv`.

### Provider detection

This course supports two backends:

| `PROVIDER` env var | What happens |
|---|---|
| `openai` | Uses the real OpenAI API. Requires `OPENAI_API_KEY`. |
| anything else (default) | Uses a local Ollama server. No API key needed. |

The code below reads these values and sets up the OpenAI client accordingly. Both backends use the same `openai` Python library — Ollama just listens on a local port and speaks the same API protocol.

In [ ]:
import os
from dotenv import load_dotenv     # reads .env file into os.environ
from openai import OpenAI          # same client for OpenAI and Ollama

# Load variables from the nearest .env file up the directory tree
load_dotenv()

# Which provider are we targeting? Defaults to 'ollama' if not set.
PROVIDER = os.getenv("PROVIDER", "ollama").lower()

# Which model should we call?
MODEL = os.getenv("MODEL", "llama3.2")

# Ollama's default local address — override with OLLAMA_BASE_URL if needed
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

# Build the right client based on the provider
if PROVIDER == "openai":
    # Real OpenAI: reads OPENAI_API_KEY from environment automatically
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
else:
    # Local Ollama: point the client at localhost, api_key is a dummy value
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

print(f"Provider : {PROVIDER}")
print(f"Model    : {MODEL}")
print("Client ready.")

## 4. The `LLMTestCase` Dataclass

### What is a test case in the world of LLMs?

In regular pytest you test a **function** with known inputs and outputs. In LLM testing you test a **conversation turn**: you sent something to the model, it replied, and now you want to evaluate that reply.

`LLMTestCase` is the container that holds everything DeepEval needs to judge one such turn.

Think of it as a **scorecard envelope** — you stuff all the relevant information in, hand it to the judge, and the judge hands it back with scores written on the outside.

### Fields at a glance

| Field | Required | What it holds | Plain-English meaning |
|---|---|---|---|
| `input` | Yes | The question / prompt you sent to your LLM | "What did you ask?" |
| `actual_output` | Yes | The response your LLM returned | "What did the AI actually say?" |
| `expected_output` | No | The ideal answer you would have wanted | "What *should* it have said?" (optional golden reference) |
| `context` | No | List of document chunks retrieved from RAG | "What source material was the AI supposed to use?" |
| `retrieval_context` | No | What the retriever actually returned | "What did the retriever fetch?" (for RAG faithfulness checks) |

> **Note:** `context` and `retrieval_context` are only needed for metrics that evaluate RAG pipelines (e.g., `FaithfulnessMetric`). For today we'll only use `input` + `actual_output`.

## 5. Create Your First Test Case

Let's build a test case manually — no real LLM call yet. We hard-code the values so we can focus on the structure before adding any moving parts.

Imagine you asked your customer-support bot: *"How do I reset my password?"* and it replied with something. We'll capture that exchange in a `LLMTestCase`.

In [ ]:
from deepeval.test_case import LLMTestCase   # the scorecard container

# Create a test case with hard-coded values (no LLM call)
manual_case = LLMTestCase(
    input="How do I reset my password?",           # the user's question
    actual_output=(
        "To reset your password, go to the login page and click "
        "'Forgot Password'. Enter your email address and we will "
        "send you a reset link within a few minutes."
    ),
    expected_output=(
        "Click 'Forgot Password' on the login page, enter your email, "
        "and follow the link we send you."
    ),
)

# Inspect the object — it's just a Python dataclass, nothing magical yet
print("Input          :", manual_case.input)
print("Actual output  :", manual_case.actual_output)
print("Expected output:", manual_case.expected_output)
print("Context        :", manual_case.context)   # None — we didn't set it

## 6. `AnswerRelevancyMetric` — The First Judge

### What does it measure?

**Answer Relevancy** asks one question: *"Does this response actually answer what was asked?"*

The judge LLM reads the `input` and the `actual_output`, then scores how closely the answer addresses the question. It does **not** check whether the answer is factually correct — just whether it's on-topic and useful.

A customer support bot that answers *"Our refund policy is 30 days"* in response to *"How do I reset my password?"* would score very low — the answer is real information, but completely irrelevant to the question asked.

### The threshold

```python
AnswerRelevancyMetric(threshold=0.7)
```

- Score range: **0.0** (completely off-topic) → **1.0** (perfectly on-point)
- `threshold=0.7` means: the test **passes** if score ≥ 0.7, **fails** if score < 0.7
- Think of it like a grading scale — 70% is the minimum passing grade

You can tighten or loosen the threshold depending on how strict your quality bar needs to be.

## 7. Measure a Test Case

Now we connect the judge to the test case. `metric.measure(case)` fires the judge LLM, waits for the score, and stores the result back on the metric object.

> **What's happening behind the scenes?** DeepEval constructs a carefully engineered prompt that contains your `input`, your `actual_output`, and a scoring rubric. It sends that prompt to the judge model and parses the JSON response to extract a numeric score and a human-readable reason.

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric   # the judge

# Create the metric with a 70% passing threshold
relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,    # score >= 0.7 → pass
    verbose_mode=True # print the judge's internal prompt & response (great for learning)
)

# Run the judge — this makes an actual LLM API call
relevancy_metric.measure(manual_case)

# The results are stored on the metric object after measuring
print("\n--- Results ---")
print(f"Score  : {relevancy_metric.score:.2f}")    # numeric score 0-1
print(f"Passed : {relevancy_metric.is_test_passed()}")  # True or False
print(f"Reason : {relevancy_metric.reason}")       # the judge's explanation

## 8. How LLM-as-a-Judge Works (Under the Hood)

Before we go further, let's demystify what just happened. This mental model will help you debug when scores look wrong.

```
┌─────────────────────────────────────────────────────────────────┐
│                      YOUR APPLICATION                           │
│                                                                 │
│   User Question                                                 │
│       │                                                         │
│       ▼                                                         │
│  ┌─────────┐     prompt      ┌───────────┐    response         │
│  │  Your   │ ─────────────►  │  Your LLM │ ──────────────┐     │
│  │  App    │                 │ (GPT/Llama│               │     │
│  └─────────┘                 └───────────┘               │     │
│                                                          ▼     │
│                                                  actual_output  │
└──────────────────────────────────────────────────────────┬─────┘
                                                           │
                              DeepEval packages:           │
                              • input (question)           │
                              • actual_output (response) ◄─┘
                              • rubric (metric definition)
                                           │
                                           ▼
                              ┌─────────────────────┐
                              │    Judge LLM        │
                              │  (GPT-4o / Llama)   │
                              │                     │
                              │  Reads the rubric,  │
                              │  compares input vs  │
                              │  output, returns:   │
                              │  • score (0–1)      │
                              │  • reason (text)    │
                              └──────────┬──────────┘
                                         │
                                         ▼
                              metric.score  / metric.reason
                              pass if score >= threshold
```

### Key insight: the judge is just another LLM

This means:
1. **Better judge = better evaluation.** GPT-4o as a judge is stricter and more reliable than a small local model.
2. **The judge can be wrong.** Always spot-check scores manually, especially edge cases.
3. **Verbosity helps.** Always read the `reason` — it tells you *why* the judge scored the way it did, which is far more useful than the number alone.
4. **Cost.** Every `metric.measure()` call = an additional LLM API call. Plan your test budget accordingly.

## 9. `assert_test()` — The pytest Integration Point

### The missing link between DeepEval and pytest

So far we've created a test case and measured it manually. But real testing pipelines run inside `pytest`, which:
- Collects all functions named `test_*`
- Runs them automatically
- Reports pass/fail in a structured way
- Integrates with CI/CD systems

`assert_test(test_case, metrics)` is the bridge. It:
1. Calls `metric.measure(test_case)` for every metric you pass
2. Raises an `AssertionError` (just like a normal `assert`) if **any** metric fails
3. Packages rich failure messages so pytest's output tells you *exactly* which metric failed and why

### Usage pattern

```python
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

def test_my_bot():
    case = LLMTestCase(
        input="What is 2+2?",
        actual_output=my_bot("What is 2+2?"),   # real LLM call here
    )
    assert_test(case, [AnswerRelevancyMetric(threshold=0.7)])
    # ↑ raises AssertionError if relevancy score < 0.7
```

> **Analogy:** `assert_test` is like the final stamp on the scorecard. The reviewer (judge LLM) has already written their comments; `assert_test` decides whether the paper gets accepted or rejected based on whether the scores met the bar.

## 10. A Real pytest Test with a Live LLM Call

Now we put it all together. We'll:
1. Write a proper pytest file to disk using `%%writefile`
2. Run it with `!pytest` so you see real pytest output inside the notebook

The test will:
- Make a real LLM call (your configured provider)
- Wrap the response in a `LLMTestCase`
- Use `assert_test` with `AnswerRelevancyMetric`

This is the exact pattern you'll use in CI pipelines.

In [ ]:
%%writefile test_deepeval_day1.py
# test_deepeval_day1.py
# Written by the notebook — run with: pytest test_deepeval_day1.py -v

import os
from dotenv import load_dotenv
from openai import OpenAI
from deepeval import assert_test                      # the pytest bridge
from deepeval.test_case import LLMTestCase            # the scorecard container
from deepeval.metrics import AnswerRelevancyMetric    # the judge metric

# ---------- Setup ----------
load_dotenv()  # load .env so keys are available

PROVIDER = os.getenv("PROVIDER", "ollama").lower()
MODEL    = os.getenv("MODEL", "llama3.2")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

# Build the right client depending on provider
if PROVIDER == "openai":
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
else:
    client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")


def call_llm(prompt: str) -> str:
    """Helper: send a prompt to the configured LLM and return the text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content  # extract the reply text


# ---------- Test ----------
def test_password_reset_is_relevant():
    """The bot's password-reset answer should be relevant to the question."""

    user_question = "How do I reset my password?"

    # Step 1: Call the real LLM to get its response
    bot_answer = call_llm(user_question)

    # Step 2: Package the exchange into a test case
    case = LLMTestCase(
        input=user_question,       # what the user asked
        actual_output=bot_answer,  # what the bot said
    )

    # Step 3: Define the quality bar — relevancy must be >= 0.7
    metric = AnswerRelevancyMetric(threshold=0.7)

    # Step 4: assert_test measures the metric and raises AssertionError if it fails
    assert_test(case, [metric])


def test_capital_city_is_relevant():
    """A geography question should get a geographically relevant answer."""

    user_question = "What is the capital city of Japan?"
    bot_answer    = call_llm(user_question)

    case = LLMTestCase(
        input=user_question,
        actual_output=bot_answer,
    )

    assert_test(case, [AnswerRelevancyMetric(threshold=0.7)])

In [ ]:
# Run the pytest file we just wrote — the -v flag gives verbose output
# -s shows any print statements inside the tests
!pytest test_deepeval_day1.py -v -s

## 11. Reading Metric Reasons

### The reason is the most valuable output

The numeric score tells you *whether* your LLM passed. The **reason** tells you *why* — and that's where the real insight lives.

When you look at a `reason`, here's what to pay attention to:

| What the reason says | What it means for your LLM |
|---|---|
| "The response directly addresses the question and provides actionable steps" | Good sign — the model is on-topic |
| "The response mentions the topic but does not answer the specific question" | The model understood the subject but drifted — tune your system prompt |
| "The response contains information unrelated to the input" | The model is hallucinating or confusing context |
| "The response is too vague to assess relevancy" | The model hedged too much — check your prompt for over-cautious instructions |
| "The response contradicts the expected output" | A factual error — might need RAG or fine-tuning |

### Reasons are better than scores for debugging

Imagine two answers that both score 0.65 (just below the 0.7 threshold). But the reasons are completely different:
- Answer A: *"Partially relevant but omits the key step about email verification"*
- Answer B: *"Addresses a related topic (account security) but not the specific password reset flow"*

The fix for Answer A is to add more detail to your prompt. The fix for Answer B is to add better context or improve retrieval. The score alone tells you nothing — the reason gives you the action.

> **Rule of thumb:** Every time a test fails, read the reason before touching any code. Often the reason reveals the fix directly.

## 12. Try It: Parametrize Over 3 Prompts

Real evaluation isn't one test — it's many. Let's loop over three different prompts, call the LLM for each, score them all, and print a mini report showing the score and reason side by side.

This is the manual version of what `pytest.mark.parametrize` does — we'll see the pytest version in Day 2.

In [ ]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric

# Three prompts that test different aspects of a hypothetical support bot
test_prompts = [
    "How do I reset my password?",
    "What are your business hours?",
    "Can you explain quantum entanglement in simple terms?",   # off-domain prompt
]

def call_llm(prompt: str) -> str:
    """Send a prompt to the configured LLM and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

print(f"{'#':<3}  {'Score':<8}  {'Pass?':<7}  Reason (truncated to 120 chars)")
print("-" * 80)

for i, prompt in enumerate(test_prompts, start=1):
    # 1. Get the LLM's response
    response = call_llm(prompt)

    # 2. Wrap in a test case
    case = LLMTestCase(input=prompt, actual_output=response)

    # 3. Score with the relevancy metric (threshold 0.7)
    metric = AnswerRelevancyMetric(threshold=0.7)
    metric.measure(case)

    # 4. Print a one-line summary
    passed_str = "PASS" if metric.is_test_passed() else "FAIL"
    reason_short = (metric.reason or "")[:120]   # trim for readability
    print(f"{i:<3}  {metric.score:<8.2f}  {passed_str:<7}  {reason_short}")
    print(f"     Prompt : {prompt}")
    print(f"     Response: {response[:100]}...")
    print()

## 13. Day 1 Summary

Here's everything we covered today in a single reference table.

| Concept | What it is | Key usage |
|---|---|---|
| **LLM-as-a-judge** | Using a second LLM to evaluate your LLM's output against a rubric | The foundation of all DeepEval metrics |
| **`LLMTestCase`** | A dataclass that holds `input`, `actual_output`, `expected_output`, `context` | `LLMTestCase(input=..., actual_output=...)` |
| **`AnswerRelevancyMetric`** | Measures whether the response answers the question asked | `AnswerRelevancyMetric(threshold=0.7)` |
| **`metric.measure(case)`** | Fires the judge LLM and stores score + reason on the metric object | Call before reading `.score` or `.reason` |
| **`metric.score`** | Float 0.0 → 1.0 from the judge | ≥ threshold = pass |
| **`metric.reason`** | The judge's plain-English explanation of the score | Read this first when debugging failures |
| **`assert_test(case, metrics)`** | pytest integration — raises `AssertionError` if any metric fails | Use inside `def test_*():` functions |
| **`%%writefile` + `!pytest`** | Write a test file from the notebook, then run pytest | Lets you demo real pytest output in a notebook |

### What's next — Day 2 preview

- `FaithfulnessMetric` — does the answer stick to the provided context, or does the model invent things?
- `ContextualPrecisionMetric` — in a RAG pipeline, how much of the retrieved context was actually used?
- `pytest.mark.parametrize` — run the same test across a dataset of inputs automatically
- Building an evaluation dataset from real user queries

---

> **Take-away from Day 1:** `assert` tests whether code is correct. `assert_test` tests whether an LLM's *language* is correct. The bridge between them is an LLM judge that reads rubrics — just like a human reviewer, but automated and reproducible.